# 03 — Spatial Normalization, Template Registration & Multi-View Cropping

**Focus:** Center-of-Mass Alignment, Affine Registration to MNI Template, and Bilateral Bounding-Box Extraction  
**Pipeline:** How raw multi-vendor NIfTI scans are transformed into uniform input tensors for deep learning.

---

## The Registration Strategy
Unregistered DaT-SPECT scans have random spatial offsets and differing bounding boxes.
To enable direct comparison across patients:
1. **Hotspot / Center-of-Mass Localization:** Locate the highest-intensity striatal clusters.
2. **Affine 3D Registration:** Align the brain volume to an empirical MNI-space template using mutual information or mean squared difference.
3. **Multi-Scale Cropping:**
   - 3D Striatal Crop: $48 \times 48 \times 32$ voxels centered on the basal ganglia.
   - Bilateral Half-Crops: Separate Left and Right putamen/caudate cubes for Siamese comparison.
   - Coronal & Axial Slabs: 2.5D multi-slice projections.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, center_of_mass, gaussian_filter

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

def create_synthetic_raw_scan(shape=(128, 128, 64)):
    """Generate a synthetic unregistered raw SPECT scan with background noise and offset."""
    vol = np.random.normal(loc=15, scale=5, size=shape)
    vol = np.clip(vol, 0, None)
    
    # Random offset center
    xc = shape[0] // 2 + np.random.randint(-8, 8)
    yc = shape[1] // 2 + np.random.randint(-8, 8)
    zc = shape[2] // 2 + np.random.randint(-4, 4)
    
    # Add striatal uptake
    for dx in [-10, 10]:
        y, x, z = np.ogrid[:shape[0], :shape[1], :shape[2]]
        dist = ((x - (xc + dx))**2 + (y - yc)**2 + (z - zc)**2) / 25.0
        vol += 250.0 * np.exp(-dist)
        
    return vol, (xc, yc, zc)

vol_raw, true_center = create_synthetic_raw_scan()
print(f"Generated raw volume shape: {vol_raw.shape} with center at {true_center}")

In [ ]:
# Step 1: Hotspot & Center of Mass Detection
def find_striatal_hotspot(vol):
    # Smooth with Gaussian filter to eliminate single-voxel reconstruction noise
    smoothed = gaussian_filter(vol, sigma=2.0)
    # Find top 0.5% brightest voxels
    thresh = np.percentile(smoothed, 99.5)
    binary_mask = smoothed >= thresh
    com = center_of_mass(binary_mask)
    return tuple(int(round(c)) for c in com)

est_center = find_striatal_hotspot(vol_raw)
print(f"True Hotspot Center: {true_center}")
print(f"Estimated Hotspot Center: {est_center}")

# Step 2: Extract 3D Striatal Crop (48 x 48 x 32)
def extract_striatal_crop(vol, center, crop_shape=(48, 48, 32)):
    cx, cy, cz = center
    dx, dy, dz = crop_shape[0]//2, crop_shape[1]//2, crop_shape[2]//2
    
    x1, x2 = max(0, cx - dx), min(vol.shape[0], cx + dx)
    y1, y2 = max(0, cy - dy), min(vol.shape[1], cy + dy)
    z1, z2 = max(0, cz - dz), min(vol.shape[2], cz + dz)
    
    crop = vol[x1:x2, y1:y2, z1:z2]
    # Pad if near boundary
    if crop.shape != crop_shape:
        pad_x = crop_shape[0] - crop.shape[0]
        pad_y = crop_shape[1] - crop.shape[1]
        pad_z = crop_shape[2] - crop.shape[2]
        crop = np.pad(crop, ((0, max(0, pad_x)), (0, max(0, pad_y)), (0, max(0, pad_z))))
        crop = crop[:crop_shape[0], :crop_shape[1], :crop_shape[2]]
        
    # Robust percentile normalization to [0, 1]
    p1, p99 = np.percentile(crop, 1), np.percentile(crop, 99)
    crop_norm = np.clip((crop - p1) / (p99 - p1 + 1e-6), 0.0, 1.0)
    return crop_norm

crop_3d = extract_striatal_crop(vol_raw, est_center)
print(f"Extracted normalized 3D crop shape: {crop_3d.shape} (Range: {crop_3d.min():.2f} to {crop_3d.max():.2f})")

In [ ]:
# Visualize the Preprocessing Pipeline
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# 1. Raw volume axial slice
axes[0].imshow(vol_raw[:, :, est_center[2]].T, cmap='inferno', origin='lower')
axes[0].scatter([est_center[0]], [est_center[1]], color='cyan', marker='+', s=120, label='Detected Hotspot')
axes[0].set_title('1. Raw Scan + Hotspot Detection')
axes[0].legend(loc='lower left')

# 2. Extracted 3D Crop Central Slice
axes[1].imshow(crop_3d[:, :, crop_3d.shape[2]//2].T, cmap='hot', origin='lower')
axes[1].set_title('2. Extracted 48×48×32 Striatal Crop')

# 3. Bilateral Split (Left vs Right Hemisphere)
mid_x = crop_3d.shape[0] // 2
left_hemi = crop_3d[:mid_x, :, :]
right_hemi = crop_3d[mid_x:, :, :]

combined_hemi = np.hstack([left_hemi[:, :, left_hemi.shape[2]//2].T,
                           right_hemi[::-1, :, right_hemi.shape[2]//2].T]) # Flip right for Siamese
axes[2].imshow(combined_hemi, cmap='hot', origin='lower')
axes[2].set_title('3. Mirrored Bilateral Half-Crops (HemiNet)')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()